# Inspect IEMOCAP

This notebook loads `AbstractTTS/IEMOCAP` from Hugging Face and inspects its structure.

It avoids decoding the `audio` column by casting it with `decode=False`, which matters in your current environment because `torchcodec` fails to load when audio decoding is triggered.

In [26]:
from datasets import Audio, load_dataset
import io
import pandas as pd
import soundfile as sf

DATASET_ID = "AbstractTTS/IEMOCAP"

In [27]:
ds = load_dataset(DATASET_ID)
ds

DatasetDict({
    train: Dataset({
        features: ['file', 'audio', 'frustrated', 'angry', 'sad', 'disgust', 'excited', 'fear', 'neutral', 'surprise', 'happy', 'EmoAct', 'EmoVal', 'EmoDom', 'gender', 'transcription', 'major_emotion', 'speaking_rate', 'pitch_mean', 'pitch_std', 'rms', 'relative_db'],
        num_rows: 10039
    })
})

## Disable audio decoding for inspection

Indexing a row on the original dataset would attempt to decode the `audio` feature. Casting that column with `decode=False` keeps the audio bytes accessible without requiring TorchCodec/FFmpeg.

In [28]:
ds_safe = ds.cast_column("audio", Audio(decode=False))
train = ds_safe["train"]

print("Rows:", train.num_rows)
print("Columns:", train.column_names)
train.features

Rows: 10039
Columns: ['file', 'audio', 'frustrated', 'angry', 'sad', 'disgust', 'excited', 'fear', 'neutral', 'surprise', 'happy', 'EmoAct', 'EmoVal', 'EmoDom', 'gender', 'transcription', 'major_emotion', 'speaking_rate', 'pitch_mean', 'pitch_std', 'rms', 'relative_db']


{'file': Value('string'),
 'audio': Audio(sampling_rate=None, decode=False, num_channels=None, stream_index=None),
 'frustrated': Value('float32'),
 'angry': Value('float32'),
 'sad': Value('float32'),
 'disgust': Value('float32'),
 'excited': Value('float32'),
 'fear': Value('float32'),
 'neutral': Value('float32'),
 'surprise': Value('float32'),
 'happy': Value('float32'),
 'EmoAct': Value('float32'),
 'EmoVal': Value('float32'),
 'EmoDom': Value('float32'),
 'gender': Value('string'),
 'transcription': Value('string'),
 'major_emotion': Value('string'),
 'speaking_rate': Value('float32'),
 'pitch_mean': Value('float32'),
 'pitch_std': Value('float32'),
 'rms': Value('float32'),
 'relative_db': Value('float32')}

In [29]:
example = train[0]
audio_info = sf.info(io.BytesIO(example["audio"]["bytes"]))

example_summary = {
    key: value
    for key, value in example.items()
    if key != "audio"
}
example_summary["audio_path"] = example["audio"]["path"]
example_summary["audio_num_bytes"] = len(example["audio"]["bytes"])
example_summary["audio_samplerate"] = audio_info.samplerate
example_summary["audio_frames"] = audio_info.frames
example_summary["audio_duration_s"] = round(audio_info.frames / audio_info.samplerate, 3)

pd.Series(example_summary)

file                Ses01F_impro01_F000.wav
frustrated                          0.00625
angry                               0.00625
sad                                 0.00625
disgust                             0.00625
excited                             0.00625
fear                                0.00625
neutral                                0.95
surprise                            0.00625
happy                               0.00625
EmoAct                             2.333333
EmoVal                             2.666667
EmoDom                                  2.0
gender                               Female
transcription                    Excuse me.
major_emotion                       neutral
speaking_rate                          5.14
pitch_mean                       202.798813
pitch_std                         76.127853
rms                                0.007884
relative_db                      -17.938435
audio_path          Ses01F_impro01_F000.wav
audio_num_bytes                 

In [30]:
preview = train.select(range(5)).remove_columns(["audio"]).to_pandas()
preview

,file,frustrated,angry,sad,disgust,excited,fear,neutral,surprise,happy,...,EmoVal,EmoDom,gender,transcription,major_emotion,speaking_rate,pitch_mean,pitch_std,rms,relative_db
0,Ses01F_impro01_F000.wav,0.006250,0.006250,0.00625,0.006250,0.00625,0.00625,0.950000,0.00625,0.00625,...,2.666667,2.000000,Female,Excuse me.,neutral,5.14,202.798813,76.127853,0.007884,-17.938435
1,Ses01F_impro01_F001.wav,0.006250,0.195000,0.00625,0.006250,0.00625,0.00625,0.761250,0.00625,0.00625,...,2.333333,2.333333,Female,Yeah.,neutral,1.45,184.518967,18.823940,0.009273,-22.341549
2,Ses01F_impro01_F002.wav,0.006250,0.195000,0.00625,0.006250,0.00625,0.00625,0.572500,0.19500,0.00625,...,2.666667,2.666667,Female,Is there a problem?,neutral,5.11,199.979996,23.654751,0.007846,-21.383230
3,Ses01F_impro01_F003.wav,0.383750,0.383750,0.00625,0.006250,0.00625,0.00625,0.195000,0.00625,0.00625,...,2.333333,3.000000,Female,You did.,neutral,4.01,175.226746,20.317553,0.013659,-18.791922
4,Ses01F_impro01_F004.wav,0.320833,0.320833,0.00625,0.163542,0.00625,0.00625,0.163542,0.00625,0.00625,...,2.666667,2.666667,Female,You were standing at the beginning and you di...,neutral,13.18,166.419769,58.883690,0.029505,-14.890097


## Label and metadata distributions

In [31]:
df = train.remove_columns(["audio"]).to_pandas()
df.head()

,file,frustrated,angry,sad,disgust,excited,fear,neutral,surprise,happy,...,EmoVal,EmoDom,gender,transcription,major_emotion,speaking_rate,pitch_mean,pitch_std,rms,relative_db
0,Ses01F_impro01_F000.wav,0.006250,0.006250,0.00625,0.006250,0.00625,0.00625,0.950000,0.00625,0.00625,...,2.666667,2.000000,Female,Excuse me.,neutral,5.14,202.798813,76.127853,0.007884,-17.938435
1,Ses01F_impro01_F001.wav,0.006250,0.195000,0.00625,0.006250,0.00625,0.00625,0.761250,0.00625,0.00625,...,2.333333,2.333333,Female,Yeah.,neutral,1.45,184.518967,18.823940,0.009273,-22.341549
2,Ses01F_impro01_F002.wav,0.006250,0.195000,0.00625,0.006250,0.00625,0.00625,0.572500,0.19500,0.00625,...,2.666667,2.666667,Female,Is there a problem?,neutral,5.11,199.979996,23.654751,0.007846,-21.383230
3,Ses01F_impro01_F003.wav,0.383750,0.383750,0.00625,0.006250,0.00625,0.00625,0.195000,0.00625,0.00625,...,2.333333,3.000000,Female,You did.,neutral,4.01,175.226746,20.317553,0.013659,-18.791922
4,Ses01F_impro01_F004.wav,0.320833,0.320833,0.00625,0.163542,0.00625,0.00625,0.163542,0.00625,0.00625,...,2.666667,2.666667,Female,You were standing at the beginning and you di...,neutral,13.18,166.419769,58.883690,0.029505,-14.890097


In [32]:
df["major_emotion"].value_counts().sort_values(ascending=False)

major_emotion
frustrated    2917
excited       1976
neutral       1726
angry         1269
sad           1250
happy          656
surprise       110
fear           107
other           26
disgust          2
Name: count, dtype: int64

In [33]:
df["gender"].value_counts().sort_values(ascending=False)

gender
Male      5239
Female    4800
Name: count, dtype: int64

In [34]:
numeric_columns = [
    "frustrated",
    "angry",
    "sad",
    "disgust",
    "excited",
    "fear",
    "neutral",
    "surprise",
    "happy",
    "EmoAct",
    "EmoVal",
    "EmoDom",
    "speaking_rate",
    "pitch_mean",
    "pitch_std",
    "rms",
    "relative_db",
]

df[numeric_columns].describe().T

,count,mean,std,min,25%,50%,75%,max
frustrated,10039.0,0.235854,0.276052,0.006250,0.006250,0.006250,0.320833,0.950000
angry,10039.0,0.141542,0.252396,0.006250,0.006250,0.006250,0.242188,0.950000
sad,10039.0,0.118023,0.242789,0.006250,0.006250,0.006250,0.006250,0.950000
disgust,10039.0,0.012361,0.039622,0.006250,0.006250,0.006250,0.006250,0.635417
excited,10039.0,0.134981,0.230411,0.006250,0.006250,0.006250,0.320833,0.950000
fear,10039.0,0.017371,0.064017,0.006250,0.006250,0.006250,0.006250,0.950000
neutral,10039.0,0.208934,0.269340,0.006250,0.006250,0.006250,0.320833,0.950000
surprise,10039.0,0.027593,0.092006,0.006250,0.006250,0.006250,0.006250,0.950000
happy,10039.0,0.103342,0.192078,0.006250,0.006250,0.006250,0.006250,0.950000
EmoAct,10039.0,3.096426,0.690732,1.000000,2.500000,3.000000,3.500000,5.000000


In [35]:
df.isna().sum().sort_values(ascending=False)

file             0
EmoVal           0
rms              0
pitch_std        0
pitch_mean       0
speaking_rate    0
major_emotion    0
transcription    0
gender           0
EmoDom           0
EmoAct           0
frustrated       0
happy            0
surprise         0
neutral          0
fear             0
excited          0
disgust          0
sad              0
angry            0
relative_db      0
dtype: int64

## Duration analysis

This reads the stored WAV bytes directly with `soundfile`, so it does not depend on TorchCodec.

In [36]:
durations_s = []

for example in train:
    info = sf.info(io.BytesIO(example["audio"]["bytes"]))
    durations_s.append(info.frames / info.samplerate)

duration_series = pd.Series(durations_s, name="duration_s")
duration_series.describe()

count    10039.000000
mean         4.460078
std          3.064676
min          0.584937
25%          2.330000
50%          3.519938
75%          5.709937
max         34.138750
Name: duration_s, dtype: float64

In [37]:
thresholds = [5, 10, 20]
duration_counts = pd.Series(
    {f"> {threshold}s": int((duration_series > threshold).sum()) for threshold in thresholds}
)
duration_counts

> 5s     3131
> 10s     603
> 20s      21
dtype: int64

In [38]:
pd.DataFrame(
    {
        "count": [int((duration_series > t).sum()) for t in thresholds],
        "fraction": [float((duration_series > t).mean()) for t in thresholds],
    },
    index=[f"> {t}s" for t in thresholds],
)

,count,fraction
> 5s,3131,0.311884
> 10s,603,0.060066
> 20s,21,0.002092


In [40]:
from IPython.display import Audio as IPyAudio, display

longest_idx = int(duration_series.idxmax())
longest_example = train[longest_idx]
longest_wav, longest_sr = sf.read(io.BytesIO(longest_example["audio"]["bytes"]))

display(pd.Series({
    "index": longest_idx,
    "file": longest_example["file"],
    "major_emotion": longest_example["major_emotion"],
    "gender": longest_example["gender"],
    "duration_s": round(duration_series.iloc[longest_idx], 3),
    "transcription": longest_example["transcription"],
}))

IPyAudio(longest_wav, rate=longest_sr)

index                                                         9797
file                                    Ses05M_script02_1_M030.wav
major_emotion                                              excited
gender                                                        Male
duration_s                                                  34.139
transcription     somewhere out there is a giant mass of silver...
dtype: object

## Optional: try decoding one audio sample

Run this only after fixing the local TorchCodec/CUDA/FFmpeg environment.

In [ ]:
# Uncomment after fixing the audio runtime.
# raw_ds = load_dataset(DATASET_ID)
# audio_example = raw_ds["train"][0]["audio"]
# audio_example